In [1]:
!wget "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz" -O Electronics.jsonl.gz

--2026-07-27 16:01:01--  https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6474438619 (6.0G) [application/gzip]
Saving to: ‘Electronics.jsonl.gz’

Electronics.jsonl.g 100%[===================>]   6.03G   105MB/s    in 80s     

2026-07-27 16:02:21 (77.1 MB/s) - ‘Electronics.jsonl.gz’ saved [6474438619/6474438619]



In [2]:
import pandas as pd
import json

# Read the downloaded compressed file directly
input_file = 'Electronics.jsonl.gz'
output_file = 'Electronics_5core.jsonl'
chunk_size = 500000 # Read 500,000 lines at a time

print("Phase 1: Scanning data to count interaction frequencies...")
user_counts = {}
item_counts = {}

# First pass: Count frequencies (Pandas can directly read .gz compressed formats)
for chunk in pd.read_json(input_file, lines=True, chunksize=chunk_size, compression='gzip'):
    for user in chunk['user_id']:
        user_counts[user] = user_counts.get(user, 0) + 1
    for item in chunk['parent_asin']:
        item_counts[item] = item_counts.get(item, 0) + 1

valid_users = {k for k, v in user_counts.items() if v >= 5}
valid_items = {k for k, v in item_counts.items() if v >= 5}

print(f"Scan complete! Found valid users: {len(valid_users)}, valid items: {len(valid_items)}")
print("Phase 2: Extracting valid data and writing to new file...")

# Second pass: Extract and save
first_chunk = True
for chunk in pd.read_json(input_file, lines=True, chunksize=chunk_size, compression='gzip'):
    cols = ['user_id', 'parent_asin', 'rating', 'timestamp', 'title', 'text']
    chunk = chunk[[c for c in cols if c in chunk.columns]]

    filtered_chunk = chunk[chunk['user_id'].isin(valid_users) & chunk['parent_asin'].isin(valid_items)]

    if not filtered_chunk.empty:
        if first_chunk:
            filtered_chunk.to_json(output_file, orient='records', lines=True)
            first_chunk = False
        else:
            filtered_chunk.to_json(output_file, orient='records', lines=True, mode='a')

print(f"Success! The compact 5-core dataset has been saved to the cloud: {output_file}")

Phase 1: Scanning data to count interaction frequencies...
Scan complete! Found valid users: 1881540, valid items: 625507
Phase 2: Extracting valid data and writing to new file...
Success! The compact 5-core dataset has been saved to the cloud: Electronics_5core.jsonl


In [ ]:
from google.colab import files
files.download('Electronics_5core.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# first 5 rows
import pandas as pd
df = pd.read_json('Electronics_5core.jsonl', lines=True, nrows=5)
print(df[['user_id', 'parent_asin', 'rating', 'text']].head())

                        user_id parent_asin  rating  \
0  AGCI7FAH4GL5FI65HYLKWTMFZ2CQ  B07CML419K       5   
1  AGCI7FAH4GL5FI65HYLKWTMFZ2CQ  B07BHHB5RH       5   
2  AGCI7FAH4GL5FI65HYLKWTMFZ2CQ  B09S6Y5BRG       5   
3  AGCI7FAH4GL5FI65HYLKWTMFZ2CQ  B01LW71IBJ       5   
4  AGCI7FAH4GL5FI65HYLKWTMFZ2CQ  B017T99JPG       5   

                                                text  
0  Light weight, quiet and totally awesome!!! It ...  
1  Update 2-they sent a new warranty replacement....  
2  These are fantastic headphones and I love that...  
3                         pretty good for the price.  
4  yes.. so good.  just buy it. my favorite featu...  


In [5]:
import pandas as pd
import numpy as np
import os

# 1. Set file paths (please modify according to your actual path in Colab)
input_file = '/content/Electronics_5core.jsonl'
output_file = 'Electronics_sampled_temp.jsonl'
chunk_size = 500000

print("==================================================")
print("Phase 1: Scanning the 7.31GB 5-core dataset to count user activity...")
print("==================================================")

user_counts = {}
chunk_count = 0

# Read in chunks, only counting user_id to save memory
for chunk in pd.read_json(input_file, lines=True, chunksize=chunk_size):
    chunk_count += 1
    print(f"  -> Processing chunk {chunk_count} ({chunk_size} rows per chunk)...")
    for user in chunk['user_id']:
        user_counts[user] = user_counts.get(user, 0) + 1

# Convert statistics to DataFrame
user_series = pd.Series(user_counts)

# Stratified labeling
conditions = [
    (user_series >= 5) & (user_series <= 10),
    (user_series >= 11) & (user_series <= 20),
    (user_series >= 21) & (user_series <= 50),
    (user_series > 50)
]
choices = ['5-10', '11-20', '21-50', '50+']
user_strata = np.select(conditions, choices, default='unknown')
user_df = pd.DataFrame({'user_id': user_series.index, 'stratum': user_strata, 'count': user_series.values})

# Filter out anomalous data (theoretically all should be >= 5)
user_df = user_df[user_df['stratum'] != 'unknown']

print("\n==================================================")
print(f"Statistics complete! Total valid 5-core users: {len(user_df)}.")
print("Real distribution proportions of groups are as follows:")
print(user_df['stratum'].value_counts(normalize=True).apply(lambda x: f"{x:.2%}"))
print("==================================================")

print("\nPhase 2: Strictly sampling 70,000 target users based on real proportions...")
# Set target user count to 70,000 to leave room for final K-core cleaning
TARGET_USERS = 70000

# Perform stratified proportional sampling, set random_state to ensure reproducible results
sampled_users_df = user_df.groupby('stratum', group_keys=False).apply(
    lambda x: x.sample(n=int(len(x) / len(user_df) * TARGET_USERS), random_state=42)
)
sampled_users_set = set(sampled_users_df['user_id'])
print(f"Successfully sampled {len(sampled_users_set)} target users!")

print("\n==================================================")
print("Phase 3: Precisely extracting data for selected users from the large file (generating a lightweight transition file)...")
print("==================================================")

first_chunk = True
chunk_count = 0

# Scan the large file again, this time extracting full data and writing to a new file
for chunk in pd.read_json(input_file, lines=True, chunksize=chunk_size):
    chunk_count += 1
    print(f"  -> Scanning chunk {chunk_count} to extract selected users...")

    # Keep only records for the selected ~35k users
    filtered_chunk = chunk[chunk['user_id'].isin(sampled_users_set)]

    if not filtered_chunk.empty:
        if first_chunk:
            filtered_chunk.to_json(output_file, orient='records', lines=True)
            first_chunk = False
        else:
            filtered_chunk.to_json(output_file, orient='records', lines=True, mode='a')

# Check the size of the generated new file
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print("\n==================================================")
print(f"Success! The transition file containing {len(sampled_users_set)} users has been saved as: {output_file}")
print(f"The file size is approximately: {file_size_mb:.2f} MB, safe to load into memory!")
print("==================================================")


Phase 1: Scanning the 7.31GB 5-core dataset to count user activity...
  -> Processing chunk 1 (500000 rows per chunk)...
  -> Processing chunk 2 (500000 rows per chunk)...
  -> Processing chunk 3 (500000 rows per chunk)...
  -> Processing chunk 4 (500000 rows per chunk)...
  -> Processing chunk 5 (500000 rows per chunk)...
  -> Processing chunk 6 (500000 rows per chunk)...
  -> Processing chunk 7 (500000 rows per chunk)...
  -> Processing chunk 8 (500000 rows per chunk)...
  -> Processing chunk 9 (500000 rows per chunk)...
  -> Processing chunk 10 (500000 rows per chunk)...
  -> Processing chunk 11 (500000 rows per chunk)...
  -> Processing chunk 12 (500000 rows per chunk)...
  -> Processing chunk 13 (500000 rows per chunk)...
  -> Processing chunk 14 (500000 rows per chunk)...
  -> Processing chunk 15 (500000 rows per chunk)...
  -> Processing chunk 16 (500000 rows per chunk)...
  -> Processing chunk 17 (500000 rows per chunk)...
  -> Processing chunk 18 (500000 rows per chunk)...
  -

/tmp/ipykernel_1178/2357428022.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_users_df = user_df.groupby('stratum', group_keys=False).apply(


Successfully sampled 69998 target users!

Phase 3: Precisely extracting data for selected users from the large file (generating a lightweight transition file)...
  -> Scanning chunk 1 to extract selected users...
  -> Scanning chunk 2 to extract selected users...
  -> Scanning chunk 3 to extract selected users...
  -> Scanning chunk 4 to extract selected users...
  -> Scanning chunk 5 to extract selected users...
  -> Scanning chunk 6 to extract selected users...
  -> Scanning chunk 7 to extract selected users...
  -> Scanning chunk 8 to extract selected users...
  -> Scanning chunk 9 to extract selected users...
  -> Scanning chunk 10 to extract selected users...
  -> Scanning chunk 11 to extract selected users...
  -> Scanning chunk 12 to extract selected users...
  -> Scanning chunk 13 to extract selected users...
  -> Scanning chunk 14 to extract selected users...
  -> Scanning chunk 15 to extract selected users...
  -> Scanning chunk 16 to extract selected users...
  -> Scanning c

In [6]:
import pandas as pd

# 1. load file
input_file = 'Electronics_sampled_temp.jsonl'
output_file = 'Electronics_final_dataset.jsonl'

df = pd.read_json(input_file, lines=True)

# 2. Execute the core iterative cleaning logic
def iterative_k_core(data, k=5):
    iteration = 1
    while True:
        start_len = len(data)

        # Filtering products
        item_counts = data['parent_asin'].value_counts()
        data = data[data['parent_asin'].isin(item_counts[item_counts >= k].index)]

        # Filtering users
        user_counts = data['user_id'].value_counts()
        data = data[data['user_id'].isin(user_counts[user_counts >= k].index)]

        if len(data) == start_len:
            print(f"Success! Convergence was achieved after {iteration} rounds of iteration.")
            break
        iteration += 1

    return data

final_dataset = iterative_k_core(df, k=5)

# 3. output
final_dataset.to_json(output_file, orient='records', lines=True)

print("\n==================================================")
print("Final dataset metrics：")
print(f"Total Reviews : {len(final_dataset)}")
print(f"Unique Users : {final_dataset['user_id'].nunique()}")
print(f"Unique Items : {final_dataset['parent_asin'].nunique()}")
print("==================================================")

Success! Convergence was achieved after 9 rounds of iteration.

Final dataset metrics：
Total Reviews : 229781
Unique Users : 27503
Unique Items : 16238


In [7]:
import pandas as pd

# 1. read data
print("Data is being read....")
file_path = '/content/Electronics_final_dataset.jsonl'
df = pd.read_json(file_path, lines=True)

# 2. sort
print("Sorting by time and user...")
df = df.sort_values(by=['user_id', 'timestamp'])

# 3. divide
print("The 80% training set and 20% testing set are currently being divided...")
def split_data(user_data):

    split_point = int(len(user_data) * 0.8)
    train = user_data.iloc[:split_point]
    test = user_data.iloc[split_point:]
    return train, test

# start dividing
grouped = df.groupby('user_id', group_keys=False)
train_list = []
test_list = []

for _, group in grouped:
    train, test = split_data(group)
    train_list.append(train)
    test_list.append(test)

# concatenate
train_data = pd.concat(train_list)
test_data = pd.concat(test_list)

# 4. save to the clooud
print("Saving the file to the cloud ...")
train_data.to_json('/content/train_data.jsonl', orient='records', lines=True)
test_data.to_json('/content/test_data.jsonl', orient='records', lines=True)

print(f"Success！training set: {len(train_data)} ，testing set: {len(test_data)} ")

Data is being read....
Sorting by time and user...
The 80% training set and 20% testing set are currently being divided...
Saving the file to the cloud ...
Success！training set: 173748 ，testing set: 56033 
